# Testing Multi-Agent Supervisor Architecture

This notebook demonstrates the multi-agent supervisor architecture using LangGraph. The system consists of a supervisor agent that coordinates specialized worker agents, starting with an SQL agent for querying the ERP database.

## Setup

First, let's install the required packages and set up our environment.

In [ ]:
# Install required packages
!pip install -r ../requirements.txt

In [ ]:
import os
import getpass

# Set OpenAI API key if not already set
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

## Generate Synthetic Data

Let's check if the synthetic data exists and generate it if needed.

In [ ]:
from agent_system.generate_data import check_database_exists, generate_synthetic_data

if not check_database_exists():
    generate_synthetic_data()
else:
    print("Database already exists. Skipping data generation.")

## Import the Multi-Agent System

Now, let's import our multi-agent system from the `agent_supervisor.py` module.

In [ ]:
from agent_system.agent_supervisor import create_multi_agent_system

# Create the multi-agent system
agent_system = create_multi_agent_system()

## Helper Function for Displaying Agent Responses

Let's create a helper function to display the agent responses in a more readable format.

In [ ]:
from langchain_core.messages import convert_to_messages
from IPython.display import display, Markdown

def pretty_print_message(message, indent=False):
    """Print a message in a readable format."""
    role = message.type
    content = message.content
    
    if role == "human":
        display(Markdown(f"**User**: {content}"))
    elif role == "ai":
        display(Markdown(f"**Agent**: {content}"))
    elif role == "system":
        display(Markdown(f"**System**: {content}"))
    elif role == "function":
        display(Markdown(f"**Function** ({message.name}): {content}"))
    else:
        display(Markdown(f"**{role}**: {content}"))

def pretty_print_messages(update, last_message=False):
    """Print messages from an agent update."""
    is_subgraph = False
    if isinstance(update, tuple):
        ns, update = update
        # skip parent graph updates in the printouts
        if len(ns) == 0:
            return

        graph_id = ns[-1].split(":")[0]
        display(Markdown(f"### Update from subgraph {graph_id}:"))
        is_subgraph = True

    for node_name, node_update in update.items():
        update_label = f"### Update from node {node_name}:"
        if is_subgraph:
            update_label = f"#### {update_label}"

        display(Markdown(update_label))

        messages = convert_to_messages(node_update["messages"])
        if last_message:
            messages = messages[-1:]

        for m in messages:
            pretty_print_message(m, indent=is_subgraph)

## Test the Multi-Agent System

Now, let's test our multi-agent system with some example queries.

### Example 1: Get Database Schema

Let's start by asking about the database schema.

In [ ]:
query = "What tables are in the ERP database and what information do they contain?"

for chunk in agent_system.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    pretty_print_messages(chunk, last_message=True)

### Example 2: Query Customer Data

Let's query some customer data from the ERP database.

In [ ]:
query = "Show me the top 5 customers by total sales amount"

for chunk in agent_system.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    pretty_print_messages(chunk, last_message=True)

### Example 3: Complex Query Across Multiple Tables

Let's try a more complex query that requires joining multiple tables.

In [ ]:
query = "Find the top 3 products by sales quantity and show their current stock levels in the warehouse"

for chunk in agent_system.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    pretty_print_messages(chunk, last_message=True)

### Example 4: Business Analysis Query

Let's try a business analysis query that requires more complex SQL.

In [ ]:
query = "What is the monthly sales trend for the past year? Show total sales amount by month."

for chunk in agent_system.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    pretty_print_messages(chunk, last_message=True)

## Interactive Testing

Finally, let's create an interactive cell where you can enter your own queries.

In [ ]:
from IPython.display import clear_output

def interactive_query():
    query = input("Enter your query (or 'exit' to quit): ")
    
    if query.lower() == "exit":
        return
    
    clear_output()
    display(Markdown(f"**User**: {query}"))
    
    for chunk in agent_system.stream(
        {"messages": [{"role": "user", "content": query}]}
    ):
        pretty_print_messages(chunk, last_message=True)
    
    # Recursively call for next query
    interactive_query()

# Start the interactive session
interactive_query()